# 06 — Final untouched test evaluation and error analysis

Evaluate selected models on held-out font families only after model selection is complete.

# Setup

Run this notebook from the repository root. In Google Colab, clone the GitHub repository first and replace the placeholder URL.

In [ ]:
from pathlib import Path
import os, sys

REPO_URL = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
if 'google.colab' in sys.modules:
    if not Path('/content/fontsense-capstone').exists():
        if 'PASTE_' in REPO_URL:
            raise ValueError('Replace REPO_URL with your GitHub repository URL first.')
        !git clone {REPO_URL} /content/fontsense-capstone
    os.chdir('/content/fontsense-capstone')
    %pip install -q -r requirements.txt
    %pip install -q -e .
else:
    root = Path.cwd()
    if root.name == 'notebooks':
        root = root.parent
    os.chdir(root)
    os.environ['PYTHONPATH'] = str(root / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
    if str(root / 'src') not in sys.path:
        sys.path.insert(0, str(root / 'src'))
print('Project root:', Path.cwd())


In [ ]:
MANIFEST = 'data/processed/fontsense_google/manifest.csv'
!python -m fontsense.evaluate --manifest {MANIFEST} --model hog
!python -m fontsense.evaluate --manifest {MANIFEST} --model cnn

In [ ]:
import json, pandas as pd
from pathlib import Path
for model in ['hog','cnn']:
    path = Path(f'reports/{model}_test_metrics.json')
    if path.exists():
        print(model.upper(), json.loads(path.read_text()))
        display(pd.read_csv(f'reports/{model}_classification_report.csv', index_col=0))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
errors = pd.read_csv('reports/hog_error_examples.csv')
if len(errors):
    sample = errors.head(10)
    fig, axes = plt.subplots(2,5,figsize=(15,6))
    for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
        ax.imshow(Image.open(row.image_path))
        ax.set_title(f"true: {row.category}\npred: {row.predicted_category}")
        ax.axis('off')
    plt.tight_layout()

Write an honest conclusion. Do not hide weak classes or difficult examples. Explain what improved performance and what the model still cannot do.